# AutoDiag AI — The Chatbot
**Capstone Project 2 · Conversational Diagnosis Assistant**

This notebook shows **how our chatbot works**, step by step.

You type a car problem, and it does three things:

| Layer | What it does |
|---|---|
| **1. Prediction** | Our trained ML model predicts the most likely fault |
| **2. Knowledge** | We look up verified causes & safety advice from our knowledge base |
| **3. Conversation** | Groq AI turns those facts into a friendly, human reply |

**The important safety idea:** the AI never diagnoses on its own. It only *rephrases* facts that come from our verified knowledge base. That is what stops it from inventing dangerous car advice.

**Files needed in this same folder:** `tfidf_vectorizer.joblib`, `fault_classifier.joblib`, `label_encoder.joblib`, `knowledge_base.csv`

## Setup — install the Groq library (run once)

In [9]:
!pip install groq

## Step 1 — Import libraries and connect to Groq
Get your free API key from **console.groq.com** → API Keys → Create API Key

In [1]:
import re                          # for cleaning the complaint text
import numpy as np                 # for finding the top predictions
import pandas as pd                # for reading the knowledge base
import joblib                      # for loading our trained model
from groq import Groq              # the AI chat library

# paste your free Groq API key between the quotes
GROQ_API_KEY = "Insert your groqapi key here"

# create the client that will talk to the Groq AI
client = Groq(api_key=GROQ_API_KEY)
print("Connected to Groq")

Connected to Groq


## Step 2 — Load our trained model and knowledge base
These 3 `.joblib` files were saved by our main pipeline notebook in Phase 7.

In [11]:
# load the three saved pieces of our trained model
tfidf = joblib.load("/Users/rohanmehta/Desktop/Capstone 2/Part1/tfidf_vectorizer.joblib")     # converts complaint text into numbers
model = joblib.load("/Users/rohanmehta/Desktop/Capstone 2/Part1/fault_classifier.joblib")     # predicts which fault it is
le    = joblib.load("/Users/rohanmehta/Desktop/Capstone 2/Part1/label_encoder.joblib")        # converts the number back to a fault name

# load the knowledge base, using the fault name as the row index so we can look it up easily
kb = pd.read_csv("/Users/rohanmehta/Desktop/Capstone 2/Part1/knowledge_base.csv").set_index("FaultCategory")

print("Loaded model for", len(le.classes_), "faults")
print("Knowledge base has", len(kb), "entries")

Loaded model for 41 faults
Knowledge base has 41 entries


## Step 3 — LAYER 1: predict the fault
We clean the text exactly the same way we did during training, then ask the model.
We return the **top 3** guesses because some faults have very similar symptoms — just like a real mechanic considers a few possibilities.

In [12]:
# clean the text the same way as in training (very important - must match!)
def clean_text(text):
    text = text.lower()                          # make everything lowercase
    text = re.sub(r"[^a-z0-9\s]", " ", text)     # replace symbols with spaces
    text = re.sub(r"\s+", " ", text)             # turn multiple spaces into one
    return text.strip()

# predict the 3 most likely faults for a complaint
def predict_faults(complaint):
    features = tfidf.transform([clean_text(complaint)])   # text -> numbers
    proba = model.predict_proba(features)[0]              # probability of each of the 41 faults
    top3 = np.argsort(proba)[::-1][:3]                    # positions of the 3 highest
    return [(le.classes_[i], proba[i]) for i in top3]     # (fault name, confidence)

# test it
predict_faults("my brake pedal feels soft and goes to the floor")

[('Brake Fluid Leak', np.float64(0.8851428365296936)),
 ('Warped Brake Rotors', np.float64(0.0214858698768948)),
 ('Clutch Wear', np.float64(0.011716010834388864))]

### Notice how confidence changes with how clear the complaint is
A **specific** complaint gives high confidence. A **vague** complaint spreads the confidence across several possible faults — the model is honestly saying *"it could be any of these"*. This is exactly what we want in a safety tool.

In [13]:
# a very specific complaint - model is confident
print("SPECIFIC complaint:")
for fault, conf in predict_faults("brake pedal feels soft and goes to the floor"):
    print(f"   {fault:32s} {conf:.1%}")

# a vague complaint - model is unsure and spreads its guesses
print("\nVAGUE complaint:")
for fault, conf in predict_faults("something feels wrong with the brakes"):
    print(f"   {fault:32s} {conf:.1%}")

SPECIFIC complaint:
   Brake Fluid Leak                 89.4%
   Warped Brake Rotors              1.9%
   Clutch Wear                      1.1%

VAGUE complaint:
   Worn Brake Pads                  20.4%
   ABS Malfunction                  19.2%
   Brake Fluid Leak                 7.8%


## Step 4 — LAYER 2: look up verified safety information
For the predicted fault, we pull real, pre-written advice from our knowledge base.
**This is the safety layer** — these facts were written from real sources, not generated by AI.

In [14]:
# get the verified details for a fault from the knowledge base
def get_fault_info(fault_name):
    row = kb.loc[fault_name]                      # find that fault's row
    return {
        "causes":        row["ProbableCauses"],      # what usually causes this fault
        "precautions":   row["PrecautionarySteps"],  # what the driver should do right now
        "action":        row["RecommendedAction"],   # the recommended repair step
        "urgency":       row["UrgencyLevel"],        # Low / Medium / High / Critical
        "safe_to_drive": row["SafeToDrive"],         # Yes / No / Caution
    }

# test it
get_fault_info("Brake Fluid Leak")

{'causes': 'Corroded brake line; failed caliper/wheel-cylinder seal; loose fitting; failing master cylinder',
 'precautions': 'Do not drive — a soft pedal can become no pedal; check reservoir level',
 'action': 'Tow to workshop; repair leak, replace fluid, bleed the system',
 'urgency': 'Critical',
 'safe_to_drive': 'No'}

## Step 5 — LAYER 3: let Groq AI write a caring reply
We give the AI two things:
1. A **system prompt** — its personality and strict rules
2. The **verified facts** from our knowledge base

The AI only rewrites those facts in friendly language. It is not allowed to invent anything.

In [15]:
# this tells the AI how to behave - caring, simple, and safety-first
SYSTEM_PROMPT = """You are AutoDiag AI, a friendly and caring car diagnosis assistant.
You will be given a diagnosis and verified facts from a trusted database.
Explain it to a worried car owner in simple, warm, reassuring language.

RULES:
- Only use the facts you are given. Never invent causes or advice.
- Start by telling them the most likely problem and how urgent it is.
- If it is not safe to drive, say so clearly and early.
- End by gently reminding them to see a professional mechanic.
- Keep it short and easy to understand. No technical jargon."""

# send the facts to Groq and get back a friendly written reply
def write_reply(complaint, fault, confidence, info):
    facts = f"""The user said: "{complaint}"

Most likely problem: {fault} (confidence {confidence:.0%})
Urgency level: {info['urgency']}
Safe to drive: {info['safe_to_drive']}
Probable causes: {info['causes']}
Precautions: {info['precautions']}
Recommended action: {info['action']}"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",       # the free Groq model
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},   # the rules
            {"role": "user",   "content": facts},           # the verified facts
        ],
        temperature=0.7,                        # a little natural variety in wording
    )
    return response.choices[0].message.content  # the AI's written answer

## Step 6 — Put all three layers together
This one function is the complete chatbot.

In [16]:
# the full chatbot: predict -> look up -> write a friendly reply
def diagnose_car(complaint):
    top3 = predict_faults(complaint)                 # LAYER 1: predict the fault
    fault, confidence = top3[0]                      # take the most likely one
    info = get_fault_info(fault)                     # LAYER 2: verified safety facts
    reply = write_reply(complaint, fault, confidence, info)   # LAYER 3: AI writes it

    # show a quick summary banner first
    print(f"DIAGNOSIS: {fault}  ({confidence:.0%} confidence)")
    print(f"URGENCY: {info['urgency']}   |   SAFE TO DRIVE: {info['safe_to_drive']}")
    print("-" * 60)
    print(reply)

    # then show the other possibilities, like a mechanic considering options
    print("-" * 60)
    print("Other possible causes:")
    for f, c in top3[1:]:
        print(f"   {f} ({c:.0%})")

## Step 7 — Try it out!

In [17]:
diagnose_car("my car shakes a lot when I stop at a signal and the engine feels rough")

DIAGNOSIS: Rough Idle  (49% confidence)
URGENCY: Medium   |   SAFE TO DRIVE: Yes
------------------------------------------------------------
Don't worry, I'm here to help you with your car concerns. It sounds like your car is experiencing a rough idle, which is likely causing the shaking when you stop at a signal. The good news is that it's not an emergency, and it's still safe to drive your car.

The rough idle might be due to a few possible causes, such as a dirty part in the engine or a small leak somewhere. It's also possible that the engine mounts are worn out or the spark plugs are old.

To help the mechanic diagnose the issue, try to notice if the problem gets worse when the engine is cold. Also, avoid pressing the accelerator slightly to try to make the engine run smoother, as this can mask the problem.

For now, I recommend scheduling a service with a professional mechanic. They can take a closer look, clean some key parts, and check the engine mounts to get your car running 

In [18]:
diagnose_car("there is a loud grinding noise when I press the brakes")

DIAGNOSIS: Worn Brake Pads  (60% confidence)
URGENCY: High   |   SAFE TO DRIVE: Caution
------------------------------------------------------------
I'm so sorry to hear that you're experiencing a loud grinding noise when you press the brakes. The most likely problem is that your brake pads are worn out, and this is considered a high-urgency issue. Unfortunately, it's not entirely safe to drive your car in this condition, so please exercise caution.

To stay safe on the road, please increase your following distance, brake gently and early, and avoid carrying heavy loads. However, I must stress that it's essential to address this issue as soon as possible. The grinding noise indicates that there's metal-on-metal damage happening, which can lead to more severe problems if not fixed promptly.

I strongly recommend that you have your brake pads replaced (and your discs checked) by a professional mechanic as soon as possible. They'll be able to assess the situation and get your car back to 

In [19]:
# type your OWN car problem here and run this cell
my_problem = "white smoke coming from the bonnet and the temperature light is on"

diagnose_car(my_problem)

DIAGNOSIS: Engine Overheating  (26% confidence)
URGENCY: Critical   |   SAFE TO DRIVE: No
------------------------------------------------------------
I'm so sorry to hear that you're experiencing white smoke coming from your bonnet and your temperature light is on. The most likely problem is that your engine is overheating, which is a critical issue that needs immediate attention. 

Please pull over to a safe location and turn off your engine right away, as it's not safe to drive your car in this condition. Driving while overheated can cause serious damage to your engine, so it's best to avoid it altogether.

It's likely that the issue is due to low coolant, a failed thermostat, or a problem with your radiator or cooling system. To be safe, do not attempt to open the radiator cap or pour cold water on the engine, as this can cause further damage.

The best course of action is to have your vehicle towed to a mechanic. They will be able to assess the situation and provide the necessary 

---
## Summary — why this design is safe

| | |
|---|---|
| **The ML model** | decides *what* the fault probably is |
| **The knowledge base** | supplies *verified* causes and safety advice |
| **The AI (Groq)** | only makes it sound friendly and human |

If we let the AI diagnose by itself, it could confidently invent wrong — and possibly dangerous — car advice. By grounding it in our knowledge base, every safety instruction the user sees came from a real, checked source.

**Next:** the same three layers run inside `app.py` as a proper web app with a chat window.